In [12]:
"""
SV callset matching with Truvari-like algorithm
modified to work with breakpoints
"""

import os

workdir = "/vast/ai_projects/Papenfuss_SVEnsemble2"
os.chdir(workdir)
os.getcwd()

import importlib
from pathlib import Path

import pandas as pd

import svensemble.vcfparse as vcfp
import svensemble._caller_params as caller_params
import svensemble.breakends as bnds
import svensemble.breakpoints as bps
import svensemble.events as evts
import svensemble.caller_specific as caller_specific
import svensemble.utils as sv_utils
import svensemble.chunk_match as chunk_match

importlib.reload(caller_params)
importlib.reload(bnds)
importlib.reload(bps)
importlib.reload(evts)
importlib.reload(caller_specific)
importlib.reload(sv_utils)
importlib.reload(chunk_match)

pd.set_option('display.max_columns', None)


samples = {}
for sample_name in ['CHM13', 'NA12878', 'NA12878_30x']:
    samples.update(sv_utils.create_config_dict(sample_name))

reference_sample_config = sv_utils.create_config_dict('gnomad', is_reference=True)

In [ ]:
for sample_name, sample_config in samples.items():
    """
    For each sample, load available breakpoint DataFrames from supported callsets,
    add the gnomAD reference breakpoints, perform outer join matching, and save results.
    """
    bp_df_dict = {}

    # Load GRIDSS breakpoints if available
    if 'gridss' in sample_config['callsets']:
        gridss_bp_path = (
            sample_config['sample_dir'] /
            'sv_calling' / 'gridss' / 'svensemble2' / 'gridss_bps.csv'
        )
        bp_df_dict['gridss'] = pd.read_csv(gridss_bp_path)

    # Load Manta candidate breakpoints if available
    if 'manta_candidate' in sample_config['callsets']:
        manta_candidate_bp_path = (
            sample_config['sample_dir'] /
            'sv_calling' / 'manta' / 'svensemble2' / 'manta_candidate_bps.csv'
        )
        bp_df_dict['manta_candidate'] = pd.read_csv(manta_candidate_bp_path)

    # Load Manta diploid breakpoints if available
    if 'manta_diploid' in sample_config['callsets']:
        manta_diploid_bp_path = (
            sample_config['sample_dir'] /
            'sv_calling' / 'manta' / 'svensemble2' / 'manta_diploid_bps.csv'
        )
        bp_df_dict['manta_diploid'] = pd.read_csv(manta_diploid_bp_path)

    # Always add gnomAD reference breakpoints
    gnomad_bp_path = (
        reference_sample_config['gnomad']['sample_dir'] /
        'svensemble2' / 'gnomad_bps.csv'
    )
    bp_df_dict['gnomad'] = pd.read_csv(gnomad_bp_path)

    # Perform breakpoint matching using chunk_match
    matched_bp_df = chunk_match.outer_join_bps(
        bp_df_dict=bp_df_dict,
        ref_callset='gnomad',
        insert_size_threshold=0.7,
        max_dist=200
    )

    # Ensure output directory exists
    output_dir = sample_config['sample_dir'] / 'sv_calling' / 'svensemble2'
    output_dir.mkdir(parents=True, exist_ok=True)

    # Save matched breakpoints to CSV
    output_csv_path = output_dir / 'matched_bps.csv'
    matched_bp_df.to_csv(output_csv_path, index=False)

    # Clean up variables to free memory
    del bp_df_dict, matched_bp_df